# Survey answers to 0/1 columns: every category kept

**Conversion to Dummy Variables** · a worked example from the StatNex teaching guide

**The story.** A streaming app surveyed 25 users about its premium plan: whether they subscribed, their age group, their gender and their main device. The table stores every answer as words. Many methods need numbers, so the categories must become 0/1 dummy columns.

**The question.** How do you turn category answers into columns a regression can use?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `12_utility/dummy_variable_conversion.csv` (25 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Premium plan,Age,Gender,Device
Subscribed,Age 18-24,Male,Phone
Subscribed,Age 18-24,Female,Computer
Subscribed,Age 18-24,Female,Phone
Not subscribed,Age 18-24,Male,Tablet
Subscribed,Age 25-29,Male,Computer
Subscribed,Age 25-29,Female,Computer
Not subscribed,Age 25-29,Male,Phone
Subscribed,Age 30-39,Male,Computer
Subscribed,Age 30-39,Female,Phone
Not subscribed,Age 30-39,Female,Tablet
Subscribed,Age 30-39,Male,Computer
Not subscribed,Age 30-39,Female,Phone
Subscribed,Age 30-39,Male,Tablet
Subscribed,Age 40-49,Female,Computer
Not subscribed,Age 40-49,Male,Tablet
Not subscribed,Age 40-49,Female,Phone
Subscribed,Age 40-49,Male,Computer
Not subscribed,Age 40-49,Female,Tablet
Not subscribed,Age 40-49,Male,Phone
Not subscribed,Age 50 or over,Female,Tablet
Subscribed,Age 50 or over,Male,Computer
Not subscribed,Age 50 or over,Female,Tablet
Not subscribed,Age 50 or over,Male,Phone
Not subscribed,Age 50 or over,Female,Computer
Not subscribed,Age 50 or over,Male,Tablet

## Libraries

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
# Each value of a variable is a category. Blank cells stay blank in every dummy column.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Premium plan", "Age", "Gender", "Device"]


def category(value):
    """A cell as category text; a number like 1.0 is written 1, as the app writes it."""
    if pd.isna(value) or str(value).strip() == "":
        return None
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value)


# Categories are put in name order the way a Japanese spreadsheet sorts: by the bytes
# of the name in the Shift-JIS (CP932) encoding, or UTF-8 for a name it cannot encode.
# Plain English names come out in alphabetical (ASCII) order.
def name_order(name):
    try:
        return name.encode("cp932")
    except UnicodeEncodeError:
        return name.encode("utf-8")

## Dummy variables

In [ ]:
# One 0/1 column for every category (k columns per variable), as principal component
# analysis and quantification method III expect. A regression cannot take them all.
dummies = pd.DataFrame({"No.": np.arange(1, len(data) + 1)})
summary = []
for name in variables:
    labels = data[name].map(category)
    present = [label for label in labels if label is not None]
    levels = sorted(set(present), key=name_order)
    kept = levels
    reference = "(none)"
    # pandas.get_dummies makes one 0/1 column per category, in the order chosen above.
    # It would give a blank row 0 in every column, which reads as "none of these", so
    # the row is set back to blank.
    coded = pd.get_dummies(pd.Categorical(labels, categories=levels), dtype=float)
    coded.loc[labels.isna().to_numpy()] = np.nan
    for level in kept:
        dummies[f"{name}:{level}"] = coded[level].to_numpy()
    summary.append([name, len(levels), len(kept), reference, ", ".join(levels)])

dummies.to_csv("dummies.csv", index=False)
print(dummies.to_string(index=False))

## Categories of each variable

In [ ]:
levels_table = pd.DataFrame(summary, columns=["variable", "categories", "columns made",
                                              "reference", "category order"])
print()
print(levels_table.to_string(index=False))

## What the result says

A dummy variable is a column that holds 1 when a user gave one answer and 0 otherwise. This run makes a column for every category: 2 for the plan, 5 for age, 2 for gender and 3 for device. The list of categories shows "(none)" as the reference category, the base that a regression compares the others with.

Now each question's columns always add up to 1. For example, user 1 has a 1 under Subscribed and a 0 under Not subscribed. One column of each question repeats what the others already say.

This full set is good for counting and for charts. A regression with a constant cannot use it, because it cannot separate columns that repeat each other. For a regression, leave one category per question out as the base.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.